# 05 — Supervised Ranking Baselines
## MerRec Balanced 2M · Popularity · MF-BPR · GRU4Rec



## Cell 1 — Imports, Config & Device


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1 — Imports, config, device
# Run after every kernel restart — everything downstream depends on this.
# ══════════════════════════════════════════════════════════════════════════════

import sys, time, random, math, json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

# ── Project root ───────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.home() / 'Desktop' / 'Capstone_C2C_Behavioral_Dynamics'
assert PROJECT_ROOT.exists(), f'Project not found: {PROJECT_ROOT}'
assert (PROJECT_ROOT / 'src').exists(), f'src/ missing in {PROJECT_ROOT}'
sys.path.insert(0, str(PROJECT_ROOT))
from src.utils.paths import P

# ── Hyperparameters ────────────────────────────────────────────────────────────
CFG = {
    # Reproducibility
    'SEED'        : 42,
    # Evaluation
    'N_NEG_EVAL'  : 99,       # negatives sampled per positive at eval time
    'TOP_K'       : 10,       # HR@K and NDCG@K
    'EVAL_BATCH'  : 512,      # users per eval batch — raise if you have RAM
    # Training
    'N_EPOCHS'    : 1,        # ← set to 20 for final submission results
    'BATCH_SIZE'  : 1024,
    'LR'          : 1e-3,
    'N_NEG_TRAIN' : 4,
    # Architecture
    'EMB_DIM'     : 64,
    'HIDDEN_DIM'  : 128,
    'MAX_SEQ_LEN' : 50,
}

# ── Device — single source of truth ───────────────────────────────────────────
DEVICE = torch.device(
    'mps'  if torch.backends.mps.is_available()  else
    'cuda' if torch.cuda.is_available()          else
    'cpu'
)

# ── Seeds ──────────────────────────────────────────────────────────────────────
random.seed(CFG['SEED'])
np.random.seed(CFG['SEED'])
torch.manual_seed(CFG['SEED'])
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(CFG['SEED'])

assert P.MERREC_BALANCED.exists(), f'Dataset not found: {P.MERREC_BALANCED}'

print(f'Project : {PROJECT_ROOT}')
print(f'Device  : {DEVICE}')
print(f'Epochs  : {CFG["N_EPOCHS"]}  (set N_EPOCHS=20 for final run)')
print(f'Metrics : HR@{CFG["TOP_K"]} · NDCG@{CFG["TOP_K"]} · AUC')
print('Cell 1 ✓')

Project : /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics
Device  : cpu
Epochs  : 1  (set N_EPOCHS=20 for final run)
Metrics : HR@10 · NDCG@10 · AUC
Cell 1 ✓


## Cell 2 — Data Load

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2 — Data load
# Reads only the three columns needed. Sorts by (user, time) for correct
# leave-one-out semantics.
# ══════════════════════════════════════════════════════════════════════════════

print('Loading parquet...')
t0 = time.time()

df = pd.read_parquet(
    P.MERREC_BALANCED,
    columns=['user_id', 'item_id', 'stime']
).sort_values(['user_id', 'stime']).reset_index(drop=True)

print(f'  {len(df):,} rows loaded in {time.time()-t0:.1f}s')

# Integer indices (0 = padding)
user2idx = {u: i+1 for i, u in enumerate(df['user_id'].unique())}
item2idx = {it: i+1 for i, it in enumerate(df['item_id'].unique())}
df['u'] = df['user_id'].map(user2idx)
df['i'] = df['item_id'].map(item2idx)

N_USERS = len(user2idx) + 1
N_ITEMS = len(item2idx) + 1

# Per-user ordered interaction lists; drop users with < 3 interactions
user_history = (
    df.groupby('u')['i']
    .apply(list)
    .to_dict()
)
user_history = {u: v for u, v in user_history.items() if len(v) >= 3}
valid_users  = list(user_history.keys())

print(f'  Users       : {N_USERS-1:,}')
print(f'  Items       : {N_ITEMS-1:,}')
print(f'  Valid users : {len(valid_users):,}  (>=3 interactions)')
print('Cell 2 ✓')

Loading parquet...
  1,924,579 rows loaded in 3.3s
  Users       : 991,427
  Items       : 1,832,240
  Valid users : 171,631  (>=3 interactions)
Cell 2 ✓


## Cell 3 — Leave-One-Out Split

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3 — Leave-one-out split
# Identical to UMAG evaluation protocol:
#   last interaction  → test positive
#   second-to-last    → val positive
#   everything else   → training sequence
# ══════════════════════════════════════════════════════════════════════════════

train_seqs    = {u: items[:-2] for u, items in user_history.items()}
val_pos       = {u: items[-2]  for u, items in user_history.items()}
test_pos      = {u: items[-1]  for u, items in user_history.items()}
user_item_set = {u: set(items) for u, items in user_history.items()}

# Pre-build item popularity from training set (used by Popularity baseline)
pop_counts = defaultdict(int)
for items in train_seqs.values():
    for it in items:
        pop_counts[it] += 1

def sample_negatives(user: int, n: int, exclude: set) -> list:
    """Fast rejection sampler. Over-samples by 3× then filters."""
    negs = []
    while len(negs) < n:
        cands = np.random.randint(1, N_ITEMS, size=n * 3)
        for c in cands:
            if c not in exclude:
                negs.append(int(c))
            if len(negs) == n:
                break
    return negs

print(f'Train : {len(train_seqs):,} users')
print(f'Val   : {len(val_pos):,} users')
print(f'Test  : {len(test_pos):,} users')
print('Cell 3 ✓')

Train : 171,631 users
Val   : 171,631 users
Test  : 171,631 users
Cell 3 ✓


## Cell 4 — Model Definitions

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4 — All model class definitions
# Kept together so a kernel restart only requires running cells 1-4
# before loading any checkpoint.
# ══════════════════════════════════════════════════════════════════════════════

# ── MF-BPR ────────────────────────────────────────────────────────────────────
class MFBPR(nn.Module):
    """Matrix Factorisation with Bayesian Personalised Ranking loss."""
    def __init__(self, n_users: int, n_items: int, emb_dim: int):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim, padding_idx=0)
        self.item_emb = nn.Embedding(n_items, emb_dim, padding_idx=0)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def forward(self, users, pos_items, neg_items):
        u  = self.user_emb(users)
        pi = self.item_emb(pos_items)
        ni = self.item_emb(neg_items)
        return -torch.log(torch.sigmoid((u*pi).sum(-1) - (u*ni).sum(-1)) + 1e-8).mean()

    def score(self, users, items):
        return (self.user_emb(users) * self.item_emb(items)).sum(-1)


# ── GRU4Rec ───────────────────────────────────────────────────────────────────
class GRU4Rec(nn.Module):
    """Session-based recommender using a GRU over item embeddings."""
    def __init__(self, n_items: int, emb_dim: int, hidden_dim: int,
                 n_layers: int = 1, dropout: float = 0.2):
        super().__init__()
        self.item_emb = nn.Embedding(n_items, emb_dim, padding_idx=0)
        self.gru      = nn.GRU(emb_dim, hidden_dim, n_layers,
                               batch_first=True,
                               dropout=dropout if n_layers > 1 else 0.0)
        self.out_proj = nn.Linear(hidden_dim, emb_dim)
        self.drop     = nn.Dropout(dropout)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def encode_seq(self, seq: torch.Tensor) -> torch.Tensor:
        """Return the GRU hidden state at the last non-padding position."""
        x        = self.drop(self.item_emb(seq))
        out, _   = self.gru(x)
        lengths  = (seq != 0).sum(dim=1).clamp(min=1) - 1
        idx      = lengths.view(-1, 1, 1).expand(-1, 1, out.size(2))
        last     = out.gather(1, idx).squeeze(1)
        return self.out_proj(last)

    def forward(self, seq, pos_items, neg_items):
        u  = self.encode_seq(seq)
        ps = (u * self.item_emb(pos_items)).sum(-1)
        ns = (u * self.item_emb(neg_items)).sum(-1)
        return -torch.log(torch.sigmoid(ps - ns) + 1e-8).mean()

    def score(self, seq, items):
        return (self.encode_seq(seq) * self.item_emb(items)).sum(-1)


print('MFBPR   defined ✓')
print('GRU4Rec defined ✓')
print('Cell 4 ✓')

MFBPR   defined ✓
GRU4Rec defined ✓
Cell 4 ✓


## Cell 5 — Shared Evaluation Engine


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5 — Shared evaluation engine
#
# evaluate_batched() accepts a score_fn with signature:
#   score_fn(users: list[int], items: list[int]) -> np.ndarray[float32]
# It receives ALL candidates for a full batch of users in one call,
# allowing vectorised scoring with no Python loop over users.
#
# This replaces the original per-user evaluate() which caused the
# 181-minute GRU4Rec evaluation time.
# ══════════════════════════════════════════════════════════════════════════════

def _pad(items: list, max_len: int) -> list:
    seq = items[-max_len:]
    return [0] * (max_len - len(seq)) + seq


def evaluate_batched(
    score_fn,
    split      : str = 'test',
    n_neg      : int = 99,
    k          : int = 10,
    batch_size : int = 512,
) -> dict:
    """
    Batched leave-one-out evaluation.
    HR@k, NDCG@k, AUC computed over (1 pos + n_neg) candidates per user.
    """
    assert split in ('val', 'test')
    pos_dict   = val_pos if split == 'val' else test_pos
    users_list = list(pos_dict.keys())
    hrs, ndcgs, aucs = [], [], []
    t0 = time.time()

    for start in range(0, len(users_list), batch_size):
        elapsed     = time.time() - t0
        done        = min(start + batch_size, len(users_list))
        pct         = 100 * done / len(users_list)
        eta         = (elapsed / done * (len(users_list) - done)) if done else 0
        print(f'\r  [{pct:5.1f}%]  {done:,}/{len(users_list):,} users'
              f'  |  elapsed {elapsed:.0f}s  eta {eta:.0f}s   ',
              end='', flush=True)

        batch_users      = users_list[start : start + batch_size]
        all_users        = []
        all_items        = []
        n_cands_per_user = []

        for u in batch_users:
            candidates = [pos_dict[u]] + sample_negatives(u, n_neg, user_item_set[u])
            all_users.extend([u] * len(candidates))
            all_items.extend(candidates)
            n_cands_per_user.append(len(candidates))

        # ONE score_fn call per batch — vectorised
        scores = score_fn(all_users, all_items)

        ptr = 0
        for n_cands in n_cands_per_user:
            s         = scores[ptr : ptr + n_cands]
            ptr      += n_cands
            labels    = np.zeros(n_cands, dtype=np.int32)
            labels[0] = 1
            rank      = int((s > s[0]).sum()) + 1
            hrs.append(  1.0 if rank <= k else 0.0)
            ndcgs.append(1.0 / math.log2(rank + 1) if rank <= k else 0.0)
            aucs.append( roc_auc_score(labels, s))

    elapsed = time.time() - t0
    print(f'\r  [100.0%]  {len(users_list):,}/{len(users_list):,} users'
          f'  |  total {elapsed:.1f}s                        ')

    return {
        f'HR@{k}'  : round(float(np.mean(hrs)),   4),
        f'NDCG@{k}': round(float(np.mean(ndcgs)), 4),
        'AUC'      : round(float(np.mean(aucs)),  4),
        'n_users'  : len(hrs),
        'split'    : split,
        'elapsed_s': round(elapsed, 1),
    }


def save_results(results: dict, name: str) -> Path:
    """Persist results JSON to results/metrics/merrec/<name>.json"""
    out_dir = P.metrics('merrec')
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f'{name}.json'
    with open(path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'  Saved → {path}')
    return path


def print_results(results: dict, model_name: str):
    k = CFG['TOP_K']
    print(f'  HR@{k}    = {results[f"HR@{k}"]:.4f}')
    print(f'  NDCG@{k}  = {results[f"NDCG@{k}"]:.4f}')
    print(f'  AUC     = {results["AUC"]:.4f}')
    print(f'  Users   = {results["n_users"]:,}')
    print(f'  Time    = {results["elapsed_s"]}s')


print('evaluate_batched() ready ✓')
print('Cell 5 ✓')

evaluate_batched() ready ✓
Cell 5 ✓


## Cell 6 — Popularity Baseline


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6 — Popularity baseline
# pop_counts was built in Cell 3 from training sequences.
# No checkpoint needed — recomputing from data is instant.
# ══════════════════════════════════════════════════════════════════════════════

def pop_score_fn(users: list, items: list) -> np.ndarray:
    return np.array([pop_counts.get(it, 0) for it in items], dtype=np.float32)

print('Evaluating Popularity...')
pop_results = evaluate_batched(
    pop_score_fn,
    split      = 'test',
    n_neg      = CFG['N_NEG_EVAL'],
    k          = CFG['TOP_K'],
    batch_size = CFG['EVAL_BATCH'],
)
pop_results['model']   = 'Popularity'
pop_results['epochs']  = 0
pop_results['dataset'] = 'merrec_balanced_2M'

print_results(pop_results, 'Popularity')
save_results(pop_results, 'popularity')
print('Cell 6 ✓')

Evaluating Popularity...
  [100.0%]  171,631/171,631 users  |  total 212.4s                        
  HR@10    = 0.0214
  NDCG@10  = 0.0171
  AUC     = 0.3536
  Users   = 171,631
  Time    = 212.4s
  Saved → /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/metrics/merrec/popularity.json
Cell 6 ✓


## Cell 7 — MF-BPR Training


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7 — MF-BPR training
# Guards against re-training if checkpoint already exists.
# ══════════════════════════════════════════════════════════════════════════════

MF_CKPT = P.ckpt('mf_bpr_merrec') / f'mf_bpr_epoch{CFG["N_EPOCHS"]}.pt'

if MF_CKPT.exists():
    print(f'Checkpoint already exists → skipping training')
    print(f'  {MF_CKPT}')
    print('Delete the file above to force re-training.')
else:
    class BPRDataset(Dataset):
        def __init__(self, train_seqs, user_item_set, n_items, n_neg=4):
            self.users         = list(train_seqs.keys())
            self.train_seqs    = train_seqs
            self.user_item_set = user_item_set
            self.n_items       = n_items
            self.n_neg         = n_neg

        def __len__(self):
            return len(self.users) * self.n_neg

        def __getitem__(self, idx):
            u   = self.users[idx % len(self.users)]
            pos = random.choice(self.train_seqs[u])
            neg = random.randint(1, self.n_items - 1)
            while neg in self.user_item_set[u]:
                neg = random.randint(1, self.n_items - 1)
            return torch.tensor(u), torch.tensor(pos), torch.tensor(neg)

    mf_model  = MFBPR(N_USERS, N_ITEMS, CFG['EMB_DIM']).to(DEVICE)
    mf_optim  = torch.optim.Adam(mf_model.parameters(), lr=CFG['LR'])
    mf_loader = DataLoader(
        BPRDataset(train_seqs, user_item_set, N_ITEMS, CFG['N_NEG_TRAIN']),
        batch_size=CFG['BATCH_SIZE'], shuffle=True, num_workers=0,
    )

    print(f'Training MF-BPR — {CFG["N_EPOCHS"]} epoch(s), {len(mf_loader):,} batches...')
    for epoch in range(CFG['N_EPOCHS']):
        mf_model.train()
        total_loss, t0 = 0.0, time.time()
        for step, (u, pos, neg) in enumerate(mf_loader):
            mf_optim.zero_grad()
            loss = mf_model(u.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE))
            loss.backward()
            mf_optim.step()
            total_loss += loss.item()

            # ── Progress line ──────────────────────────────────────────────
            done    = step + 1
            total   = len(mf_loader)
            pct     = 100 * done / total
            elapsed = time.time() - t0
            eta     = (elapsed / done) * (total - done)
            print(f'\r  Epoch {epoch+1}/{CFG["N_EPOCHS"]}  '
                  f'[{done:,}/{total:,}  {pct:5.1f}%]  '
                  f'loss {total_loss/done:.4f}  '
                  f'elapsed {elapsed:.0f}s  eta {eta:.0f}s   ',
                  end='', flush=True)

        print(f'\r  Epoch {epoch+1}/{CFG["N_EPOCHS"]}  '
              f'[{len(mf_loader):,}/{len(mf_loader):,}  100.0%]  '
              f'loss {total_loss/len(mf_loader):.4f}  '
              f'elapsed {time.time()-t0:.1f}s  eta 0s   ')

Training MF-BPR — 1 epoch(s), 671 batches...
  Epoch 1/1  [671/671  100.0%]  loss 0.6859  elapsed 2803.6s  eta 0s    


## Cell 8 — MF-BPR Evaluation


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8 — MF-BPR evaluation from checkpoint
# ══════════════════════════════════════════════════════════════════════════════

MF_CKPT = P.ckpt('mf_bpr_merrec') / f'mf_bpr_epoch{CFG["N_EPOCHS"]}.pt'
assert MF_CKPT.exists(), f'Checkpoint not found: {MF_CKPT}\nRun Cell 7 first.'

mf_model = MFBPR(N_USERS, N_ITEMS, CFG['EMB_DIM']).to(DEVICE)
mf_model.load_state_dict(torch.load(MF_CKPT, map_location=DEVICE))
mf_model.eval()
print(f'Loaded: {MF_CKPT}')

def mf_score_fn(users: list, items: list) -> np.ndarray:
    with torch.no_grad():
        u = torch.tensor(users, dtype=torch.long, device=DEVICE)
        i = torch.tensor(items, dtype=torch.long, device=DEVICE)
        return mf_model.score(u, i).cpu().numpy()

print('Evaluating MF-BPR...')
mf_results = evaluate_batched(
    mf_score_fn,
    split      = 'test',
    n_neg      = CFG['N_NEG_EVAL'],
    k          = CFG['TOP_K'],
    batch_size = CFG['EVAL_BATCH'],
)
mf_results['model']      = 'MF-BPR'
mf_results['epochs']     = CFG['N_EPOCHS']
mf_results['dataset']    = 'merrec_balanced_2M'
mf_results['checkpoint'] = str(MF_CKPT)

print_results(mf_results, 'MF-BPR')
save_results(mf_results, f'mf_bpr_epoch{CFG["N_EPOCHS"]}')
print('Cell 8 ✓')

Loaded: /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/checkpoints/mf_bpr_merrec/mf_bpr_epoch1.pt
Evaluating MF-BPR...
  [100.0%]  171,631/171,631 users  |  total 342.9s                        
  HR@10    = 0.0765
  NDCG@10  = 0.0331
  AUC     = 0.5016
  Users   = 171,631
  Time    = 342.9s
  Saved → /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/metrics/merrec/mf_bpr_epoch1.json
Cell 8 ✓


## Cell 9 — GRU4Rec Training


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9 — GRU4Rec training
# Guards against re-training if checkpoint already exists.
# ══════════════════════════════════════════════════════════════════════════════

GRU_CKPT = P.ckpt('gru4rec_merrec') / f'gru4rec_epoch{CFG["N_EPOCHS"]}.pt'

if GRU_CKPT.exists():
    print(f'Checkpoint already exists → skipping training')
    print(f'  {GRU_CKPT}')
    print('Delete the file above to force re-training.')
else:
    class GRU4RecDataset(Dataset):
        def __init__(self, train_seqs, user_item_set, n_items, max_len, n_neg=4):
            self.users         = list(train_seqs.keys())
            self.train_seqs    = train_seqs
            self.user_item_set = user_item_set
            self.n_items       = n_items
            self.max_len       = max_len
            self.n_neg         = n_neg

        def __len__(self):
            return len(self.users) * self.n_neg

        def _pad(self, items):
            seq = items[-self.max_len:]
            return torch.tensor(
                [0] * (self.max_len - len(seq)) + seq, dtype=torch.long
            )

        def __getitem__(self, idx):
            u     = self.users[idx % len(self.users)]
            items = self.train_seqs[u]
            t     = random.randint(1, len(items) - 1) if len(items) >= 2 else 0
            seq   = self._pad(items[:t])
            pos   = items[t]
            neg   = random.randint(1, self.n_items - 1)
            while neg in self.user_item_set[u]:
                neg = random.randint(1, self.n_items - 1)
            return seq, torch.tensor(pos), torch.tensor(neg)

    gru_model  = GRU4Rec(N_ITEMS, CFG['EMB_DIM'], CFG['HIDDEN_DIM']).to(DEVICE)
    gru_optim  = torch.optim.Adam(gru_model.parameters(), lr=CFG['LR'])
    gru_loader = DataLoader(
        GRU4RecDataset(train_seqs, user_item_set, N_ITEMS,
                       CFG['MAX_SEQ_LEN'], CFG['N_NEG_TRAIN']),
        batch_size=CFG['BATCH_SIZE'], shuffle=True, num_workers=0,
    )

    print(f'Training GRU4Rec — {CFG["N_EPOCHS"]} epoch(s), {len(gru_loader):,} batches...')
    for epoch in range(CFG['N_EPOCHS']):
        gru_model.train()
        total_loss, t0 = 0.0, time.time()
        for step, (seqs, pos, neg) in enumerate(gru_loader):
            gru_optim.zero_grad()
            loss = gru_model(seqs.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gru_model.parameters(), 1.0)
            gru_optim.step()
            total_loss += loss.item()

            # ── Progress line ──────────────────────────────────────────────
            done    = step + 1
            total   = len(gru_loader)
            pct     = 100 * done / total
            elapsed = time.time() - t0
            eta     = (elapsed / done) * (total - done)
            print(f'\r  Epoch {epoch+1}/{CFG["N_EPOCHS"]}  '
                  f'[{done:,}/{total:,}  {pct:5.1f}%]  '
                  f'loss {total_loss/done:.4f}  '
                  f'elapsed {elapsed:.0f}s  eta {eta:.0f}s   ',
                  end='', flush=True)

        print(f'\r  Epoch {epoch+1}/{CFG["N_EPOCHS"]}  '
              f'[{len(gru_loader):,}/{len(gru_loader):,}  100.0%]  '
              f'loss {total_loss/len(gru_loader):.4f}  '
              f'elapsed {time.time()-t0:.1f}s  eta 0s   ')

Checkpoint already exists → skipping training
  /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/checkpoints/gru4rec_merrec/gru4rec_epoch1.pt
Delete the file above to force re-training.


## Cell 10 — GRU4Rec Evaluation


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 10 — GRU4Rec evaluation from checkpoint
#
# Key fix vs original notebook:
# score_fn receives ALL candidates for a full batch of users at once.
# Sequences are padded and stacked into one tensor → single forward pass.
# This is why eval time drops from ~180 min to ~3 min.
# ══════════════════════════════════════════════════════════════════════════════

GRU_CKPT = P.ckpt('gru4rec_merrec') / f'gru4rec_epoch{CFG["N_EPOCHS"]}.pt'
assert GRU_CKPT.exists(), f'Checkpoint not found: {GRU_CKPT}\nRun Cell 9 first.'

gru_model = GRU4Rec(N_ITEMS, CFG['EMB_DIM'], CFG['HIDDEN_DIM']).to(DEVICE)
gru_model.load_state_dict(torch.load(GRU_CKPT, map_location=DEVICE))
gru_model.eval()
print(f'Loaded: {GRU_CKPT}')

# Build a user → padded sequence lookup once (avoids repeated _pad calls)
user_seq_cache = {
    u: _pad(train_seqs.get(u, [1]), CFG['MAX_SEQ_LEN'])
    for u in test_pos.keys()
}

def gru_score_fn(users: list, items: list) -> np.ndarray:
    """
    users and items are parallel lists of length sum(candidates per user).
    Each user id appears len(candidates) times consecutively.
    We look up the pre-padded sequence for each user and score in one pass.
    """
    seqs = [user_seq_cache.get(u, [0] * CFG['MAX_SEQ_LEN']) for u in users]
    with torch.no_grad():
        seqs_t  = torch.tensor(seqs,  dtype=torch.long, device=DEVICE)
        items_t = torch.tensor(items, dtype=torch.long, device=DEVICE)
        return gru_model.score(seqs_t, items_t).cpu().numpy()

print('Evaluating GRU4Rec...')
gru_results = evaluate_batched(
    gru_score_fn,
    split      = 'test',
    n_neg      = CFG['N_NEG_EVAL'],
    k          = CFG['TOP_K'],
    batch_size = CFG['EVAL_BATCH'],
)
gru_results['model']      = 'GRU4Rec'
gru_results['epochs']     = CFG['N_EPOCHS']
gru_results['dataset']    = 'merrec_balanced_2M'
gru_results['checkpoint'] = str(GRU_CKPT)

print_results(gru_results, 'GRU4Rec')
save_results(gru_results, f'gru4rec_epoch{CFG["N_EPOCHS"]}')
print('Cell 10 ✓')

Loaded: /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/checkpoints/gru4rec_merrec/gru4rec_epoch1.pt
Evaluating GRU4Rec...
  [100.0%]  171,631/171,631 users  |  total 21599.5s                        
  HR@10    = 0.0437
  NDCG@10  = 0.0166
  AUC     = 0.4797
  Users   = 171,631
  Time    = 21599.5s
  Saved → /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/metrics/merrec/gru4rec_epoch1.json
Cell 10 ✓


## Cell 11 — Validation Sweep (Optional)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 11 — Validation evaluation (optional)
# Useful for epoch selection before the final run.
# Requires cells 6, 8, 10 to have been run (score_fn closures in scope).
# ══════════════════════════════════════════════════════════════════════════════

print('Validation sweep...')
val_results = {}

for name, fn in [('Popularity', pop_score_fn),
                 ('MF-BPR',     mf_score_fn),
                 ('GRU4Rec',    gru_score_fn)]:
    print(f'  {name}...')
    r = evaluate_batched(fn, split='val',
                         n_neg=CFG['N_NEG_EVAL'], k=CFG['TOP_K'],
                         batch_size=CFG['EVAL_BATCH'])
    r['model'] = name
    val_results[name] = r
    save_results(r, f'{name.lower().replace("-","_")}_val_epoch{CFG["N_EPOCHS"]}')

k = CFG['TOP_K']
print(f'\n{"Model":<14} {"HR@10":>8} {"NDCG@10":>10} {"AUC":>8}  (val)')
print('─' * 46)
for name, r in val_results.items():
    print(f'{name:<14} {r[f"HR@{k}"]:>8.4f} {r[f"NDCG@{k}"]:>10.4f} {r["AUC"]:>8.4f}')
print('Cell 11 ✓')

Validation sweep...
  Popularity...
  [100.0%]  171,631/171,631 users  |  total 535.8s                        
  Saved → /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/metrics/merrec/popularity_val_epoch1.json
  MF-BPR...
  [100.0%]  171,631/171,631 users  |  total 621.6s                        
  Saved → /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/metrics/merrec/mf_bpr_val_epoch1.json
  GRU4Rec...
  [ 48.3%]  82,944/171,631 users  |  elapsed 10783s  eta 11530s   

## Cell 12 — Final Results Table


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 12 — Results table
# Loads all results from disk — independent of kernel state.
# UMAG numbers: set manually once 09_umag_merrec.ipynb has been run,
#               or they will be read automatically if the JSON exists.
# ══════════════════════════════════════════════════════════════════════════════

import json, math
from pathlib import Path

METRICS_DIR = P.metrics('merrec')

def load_result(filename: str) -> dict | None:
    path = METRICS_DIR / filename
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None

# ── Load all baseline results ──────────────────────────────────────────────────
EPOCH = CFG['N_EPOCHS']   # change if you want to compare a different epoch

results = {
    'Popularity' : load_result('popularity.json'),
    'MF-BPR'     : load_result(f'mf_bpr_epoch{EPOCH}.json'),
    'GRU4Rec'    : load_result(f'gru4rec_epoch{EPOCH}.json'),
    'UMAG v6'    : load_result(f'umag_epoch{EPOCH}.json'),   # populated by 09_umag_merrec.ipynb
}

k = CFG['TOP_K']

def _fmt(v, key):
    if v is None: return 'pending'
    val = v.get(key)
    if val is None or (isinstance(val, float) and math.isnan(val)): return 'pending'
    return f'{val:.4f}'

# ── Print table ────────────────────────────────────────────────────────────────
notes = {
    'Popularity' : 'no personalization, no history',
    'MF-BPR'     : 'no sequential history, no gate',
    'GRU4Rec'    : 'no gate, no tier-awareness',
    'UMAG v6'    : 'full model  ← target',
}

print(f'Epochs  : {EPOCH}  (set N_EPOCHS=20 for final submission)')
print(f'Dataset : merrec_balanced_2M')
print(f'Protocol: leave-one-out · {CFG["N_NEG_EVAL"]} negatives · HR@{k} · NDCG@{k} · AUC')
print()
print(f'{"Model":<14} {f"HR@{k}":>8} {f"NDCG@{k}":>10} {"AUC":>8}  Note')
print('─' * 74)
for name, r in results.items():
    print(f'{name:<14} {_fmt(r, f"HR@{k}"):>8} {_fmt(r, f"NDCG@{k}"):>10}'
          f' {_fmt(r, "AUC"):>8}  {notes[name]}')
print('─' * 74)

# ── Save combined CSV ──────────────────────────────────────────────────────────
rows = []
for name, r in results.items():
    rows.append({
        'model'  : name,
        f'HR@{k}': None if r is None else r.get(f'HR@{k}'),
        f'NDCG@{k}': None if r is None else r.get(f'NDCG@{k}'),
        'AUC'    : None if r is None else r.get('AUC'),
        'epochs' : EPOCH,
        'dataset': 'merrec_balanced_2M',
    })

out_csv = METRICS_DIR / f'all_baselines_epoch{EPOCH}.csv'
pd.DataFrame(rows).to_csv(out_csv, index=False)
print(f'\nSaved combined CSV → {out_csv}')
print('Cell 12 ✓')

Epochs  : 1  (set N_EPOCHS=20 for final submission)
Dataset : merrec_balanced_2M
Protocol: leave-one-out · 99 negatives · HR@10 · NDCG@10 · AUC

Model             HR@10    NDCG@10      AUC  Note
──────────────────────────────────────────────────────────────────────────
Popularity       0.0214     0.0171   0.3536  no personalization, no history
MF-BPR           0.0765     0.0331   0.5016  no sequential history, no gate
GRU4Rec          0.0437     0.0166   0.4797  no gate, no tier-awareness
UMAG v6         pending    pending  pending  full model  ← target
──────────────────────────────────────────────────────────────────────────

Saved combined CSV → /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/metrics/merrec/all_baselines_epoch1.csv
Cell 12 ✓
